# Batch Scoring — Delivery Delay Prediction

Scores every delivery; writes predictions + per-depot delay risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/delay_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'planned_duration_hrs', 'distance_km', 'load_tonnes', 'load_utilisation',
    'sla_hours', 'toll_cost_gbp', 'capacity_tonnes', 'vehicle_age',
    'departure_hour', 'departure_dow', 'is_weekend', 'is_rush',
]
cat_cols = ['vehicle_type', 'depot', 'route_type']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('delay_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_late', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('delay_probability') > 0.8, 'critical')
        .when(col('delay_probability') > 0.6, 'high')
        .when(col('delay_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'delivery_id', 'vehicle_id', 'route_id', 'vehicle_type', 'depot', 'route_type',
        'distance_km', 'load_utilisation', 'planned_duration_hrs',
        'had_late', 'predicted_late', 'delay_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-depot delay risk summary
summary = (
    predictions
    .groupBy('depot')
    .agg(
        count('*').alias('total_deliveries'),
        spark_sum('predicted_late').alias('predicted_late_count'),
        spark_sum('had_late').alias('actual_late_count'),
        spark_round(avg('delay_probability'), 4).alias('avg_delay_probability'),
        spark_round(avg('distance_km'), 1).alias('avg_distance_km'),
        spark_round(avg('load_utilisation'), 3).alias('avg_load_utilisation'),
    )
    .withColumn('late_rate', spark_round(col('predicted_late_count') / col('total_deliveries') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_delay_probability') > 0.6, 'high')
        .when(col('avg_delay_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_delay_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Depot delay summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')